# Checkerboard one-image FRC - recalibration against the two-image gold standard

Recalibrates the **checkerboard single-image FRC** , following
the procedure of **Dumoux et al. 2023** 



In [ ]:
DATA_ROOT   = r"../data/frc_calibration"
OUTPUT_PATH = r"../outputs/checkerboard_calibration"   # "" to skip CSV output

# --- geometry ------------------------------------------------------------------------------
# Every pair is centre-cropped to the same CROP_SIZE after registration. A fixed size (rather
# than one that depends on each pair's drift) keeps the FRC ring binning identical across pairs,
# and 2048 = 8 x 256 tiles exactly into the patch grid with no remainder.
CROP_SIZE  = 2048
PATCH_SIZE = 256

# --- registration --------------------------------------------------------------------------
UPSAMPLE        = 50     # phase-correlation precision = 1/UPSAMPLE px
MAX_RESIDUAL_PX = 0.1    # post-registration residual above this is flagged

# --- verification (both off by default) ------------------------------------------------------
INSPECT_IN_NAPARI = False   # open napari and check registration by eye
SHOW_QUANT_CHECKS = False   # difference-image statistics for every pair
INSPECT_PAIR_IDX  = 0       # which pair the preview and napari cells use

# --- FRC (used from Step 3 onward) -----------------------------------------------------------
THRESHOLD = "0.143"   # fixed threshold used by both Koho and Dumoux
N_RINGS   = None      # None -> min(image_shape) // 2

# No edge taper. miplib, and so Quoll, applies a Hamming window before the checkerboard split;
# Step 3 sets out why that is the wrong trade here. Set "hann" or "hamming" to test sensitivity.
FRC_WINDOW  = "none"
SMOOTH_FRAC = 0.02    # root-finding smoothing width, as a fraction of the number of rings

# Checkerboard sampling limit: the two halves sit on a lattice of twice the pixel spacing, so
# their Nyquist period is 4 original pixels. Nothing finer is measurable - a physical limit,
# not a free parameter. At 5 nm/px that is a 20 nm limit.
CHECKERBOARD_FLOOR_PX = 4.0

# Nothing is withheld from the fit for sitting close to that limit. Dumoux et al. define no such
# exclusion, and the calibration exists precisely to correct the regime where the one-image bias
# is largest, so dropping those points would mean fitting the easy end and extrapolating into the
# hard one. A crossing found within this multiple of the limit is only *flagged*: 1.05 is the
# width of the numerical edge effects (the smoothing window reflects there and the persistence
# rule runs out of axis). Step 8 refits without the flagged points and reports the difference.
NYQUIST_FLAG_MULTIPLE = 1.05

RANDOM_SEED = 0

print("Data root      :", DATA_ROOT)
print("Output         :", OUTPUT_PATH or "(none)")
print("Crop / patch   : %d px -> %d x %d patches of %d px"
      % (CROP_SIZE, CROP_SIZE // PATCH_SIZE, CROP_SIZE // PATCH_SIZE, PATCH_SIZE))
print("Registration   : phase cross-correlation at 1/%d px" % UPSAMPLE)
print("Verification   : napari %s | quantitative %s"
      % ("on" if INSPECT_IN_NAPARI else "off", "on" if SHOW_QUANT_CHECKS else "off"))
print("Threshold      :", THRESHOLD)
print("FRC window     : %s | root-finding on a curve smoothed over %.0f%% of the rings"
      % (FRC_WINDOW, 100 * SMOOTH_FRAC))
print("Sampling limit : %.1f px (checkerboard Nyquist period); flagged, not dropped, below "
      "%.2f px" % (CHECKERBOARD_FLOOR_PX, NYQUIST_FLAG_MULTIPLE * CHECKERBOARD_FLOOR_PX))

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from skimage.registration import phase_cross_correlation

pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

## Step 1 - discover repeat pairs and read acquisition metadata

Walk `DATA_ROOT` for `*_1.tif` files that have a matching `*_2.tif` sibling. For each pair, read
the **pixel size, HV, beam current and dwell time from the embedded FEI metadata**
(`FEI_HELIOS` TIFF tag). That is the ground truth for physical units, so nothing here depends on
folder names or hand-typed constants.

Expected: 15 pairs - 5 ROIs x 3 pixel sizes (1, 2.5, 5 nm).

In [ ]:
def read_fei_metadata(path):
    """Pixel size / HV / current / dwell from the FEI_HELIOS TIFF tag (NaN if absent)."""
    out = {"px_nm": np.nan, "HV_kV": np.nan, "current_pA": np.nan, "dwell_us": np.nan}
    try:
        with tifffile.TiffFile(path) as tf:
            tag = tf.pages[0].tags.get("FEI_HELIOS")
            if tag is None:
                return out
            scan = tag.value.get("Scan", {})
            ebeam = tag.value.get("EBeam", {})
            out["px_nm"] = float(scan.get("PixelWidth")) * 1e9
            out["HV_kV"] = float(ebeam.get("HV")) / 1000.0
            out["current_pA"] = float(ebeam.get("BeamCurrent")) * 1e12
            out["dwell_us"] = float(scan.get("Dwelltime")) * 1e6
    except (TypeError, ValueError):
        pass
    return out


def load_frame(path):
    return tifffile.imread(path).astype(np.float64)


PAIRS = []
for dirpath, _, filenames in os.walk(DATA_ROOT):
    for fname in sorted(filenames):
        if not fname.lower().endswith("_1.tif"):
            continue
        sibling = fname[:-6] + "_2.tif"
        if sibling not in filenames:
            continue
        p1 = os.path.join(dirpath, fname)
        PAIRS.append({
            "label": fname[:-6],
            "group": os.path.basename(dirpath),
            "path1": p1,
            "path2": os.path.join(dirpath, sibling),
            **read_fei_metadata(p1),
        })

if not PAIRS:
    raise RuntimeError(f"No *_1.tif / *_2.tif pairs found under {DATA_ROOT}")

PAIRS.sort(key=lambda p: (p["px_nm"], p["label"]))

with tifffile.TiffFile(PAIRS[0]["path1"]) as tf:
    FRAME_SHAPE = tf.pages[0].shape

print(f"Found {len(PAIRS)} repeat pairs | frame shape {FRAME_SHAPE}")
if min(FRAME_SHAPE) < CROP_SIZE:
    raise ValueError(f"CROP_SIZE={CROP_SIZE} exceeds the frame size {FRAME_SHAPE}")
if CROP_SIZE % PATCH_SIZE:
    print(f"NOTE: CROP_SIZE {CROP_SIZE} is not a multiple of PATCH_SIZE {PATCH_SIZE}; "
          f"the patch grid will leave a remainder")
print(f"Centre crop: {min(FRAME_SHAPE)} -> {CROP_SIZE} px, trimming "
      f"{(min(FRAME_SHAPE) - CROP_SIZE) // 2} px off each side "
      f"({100 * (1 - CROP_SIZE ** 2 / min(FRAME_SHAPE) ** 2):.0f}% of the pixels). "
      f"The trim is what removes the phase-ramp wrap-around; see Step 2.")

pairs_df = pd.DataFrame([{k: p[k] for k in ("group", "label", "px_nm", "HV_kV", "current_pA", "dwell_us")}
                         for p in PAIRS])
pairs_df

### Visual check

One pair side by side, plus their **difference image**. Before registration the difference shows
edge "ghosting" wherever the frames are shifted. Change `INSPECT_PAIR_IDX` to look at other pairs.

In [ ]:
pair = PAIRS[INSPECT_PAIR_IDX]
a = load_frame(pair["path1"])
b = load_frame(pair["path2"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, ttl in zip(axes, [a, b, a - b],
                        [f"{pair['label']} _1", f"{pair['label']} _2", "difference (unregistered)"]):
    lo, hi = np.percentile(img, [1, 99])
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(ttl, fontsize=10)
    ax.axis("off")
fig.suptitle(f"{pair['group']}  -  {pair['px_nm']:.1f} nm/px", fontsize=11)
fig.tight_layout()
plt.show()

print(f"frame 1: mean={a.mean():.0f}  std={a.std():.0f}   frame 2: mean={b.mean():.0f}  std={b.std():.0f}")
print(f"difference std: {(a - b).std():.0f}  "
      f"(per-frame noise ~ {((a - b).std() / np.sqrt(2)):.0f} once registered)")

## Step 2 - register each pair of same ROI

In [ ]:
def centre_crop(img, n=None):
    """Central n x n crop, so every pair ends up with identical geometry."""
    n = CROP_SIZE if n is None else n
    y0 = (img.shape[0] - n) // 2
    x0 = (img.shape[1] - n) // 2
    if y0 < 0 or x0 < 0:
        raise ValueError(f"crop {n} larger than image {img.shape}")
    return img[y0:y0 + n, x0:x0 + n]


def measure_shift(ref, mov, upsample=None):
    """Sub-pixel (dy, dx) that must be applied to *mov* for it to align with *ref*."""
    shift, _, _ = phase_cross_correlation(ref, mov, upsample_factor=upsample or UPSAMPLE)
    return np.asarray(shift, dtype=np.float64)


def fourier_shift_image(img, shift):
    """Translate by (dy, dx) with a Fourier phase ramp - exact for a pure translation."""
    ny, nx = img.shape
    fy = np.fft.fftfreq(ny)[:, None]
    fx = np.fft.fftfreq(nx)[None, :]
    F = np.fft.fft2(img) * np.exp(-2j * np.pi * (shift[0] * fy + shift[1] * fx))
    return np.real(np.fft.ifft2(F))


def register_pair(a, b, max_iter=4, tol=None):
    """Shift aligning *b* to *a*, refined on the cropped region that the FRC will actually use.

    The first estimate comes from the full frames, but it is applied to and verified on the
    CROP_SIZE centre. Those are different pixels, so the full-frame optimum is not quite the
    crop's optimum. Re-measuring on the crop and accumulating the residual closes that gap;
    translations compose additively, so the correction is just a sum.
    """
    tol = MAX_RESIDUAL_PX if tol is None else tol
    a_c = centre_crop(a)
    shift = measure_shift(a, b)

    for n_iter in range(1, max_iter + 1):
        residual = measure_shift(a_c, centre_crop(fourier_shift_image(b, shift)))
        if np.max(np.abs(residual)) < tol:
            return shift, residual, n_iter
        shift = shift + residual

    residual = measure_shift(a_c, centre_crop(fourier_shift_image(b, shift)))
    return shift, residual, max_iter


def get_registered(pair, n=None):
    """Load a pair, apply its cached shift to frame 2, centre-crop both to the same size.

    Frames are re-read rather than cached: 15 pairs of 2048^2 float64 would be ~1 GB.
    """
    a = load_frame(pair["path1"])
    b = fourier_shift_image(load_frame(pair["path2"]), SHIFTS[pair["label"]])
    return centre_crop(a, n), centre_crop(b, n)


print("Registration functions ready.")

In [ ]:
SHIFTS = {}
reg_rows = []

for pair in PAIRS:
    a = load_frame(pair["path1"])
    b = load_frame(pair["path2"])

    shift, residual, n_iter = register_pair(a, b)
    SHIFTS[pair["label"]] = shift

    reg_rows.append({
        "group": pair["group"], "label": pair["label"], "px_nm": pair["px_nm"],
        "shift_y": shift[0], "shift_x": shift[1],
        "drift_px": float(np.hypot(*shift)),
        "drift_nm": float(np.hypot(*shift)) * pair["px_nm"],
        "residual_y": residual[0], "residual_x": residual[1],
        "residual_px": float(np.hypot(*residual)),
        "iterations": n_iter,
    })

    flag = "" if np.max(np.abs(residual)) < MAX_RESIDUAL_PX else "  <-- CHECK"
    print(f"{pair['label']:28s} shift=({shift[0]:+6.2f}, {shift[1]:+6.2f}) px   "
          f"residual=({residual[0]:+5.2f}, {residual[1]:+5.2f})   {n_iter} iter{flag}")

reg_df = pd.DataFrame(reg_rows)

worst = reg_df["residual_px"].max()
n_flagged = int((reg_df[["residual_y", "residual_x"]].abs().max(axis=1) >= MAX_RESIDUAL_PX).sum())
print(f"\nMax |residual|: {worst:.3f} px   |   {n_flagged}/{len(reg_df)} pairs above "
      f"{MAX_RESIDUAL_PX} px")
print(f"Drift range   : {reg_df['drift_px'].min():.2f} - {reg_df['drift_px'].max():.2f} px "
      f"({reg_df['drift_nm'].min():.1f} - {reg_df['drift_nm'].max():.1f} nm)")

reg_df

### Check registration by eye (napari)

Set `INSPECT_IN_NAPARI = True` in the configuration cell and re-run from there.

The viewer opens with the pair stacked into one layer, so dragging the top slider **blinks**
between frame 1 and frame 2. That is the check: if registration is right, nothing moves. A second
layer holds the unregistered pair for comparison, and two difference layers are there if you want
them. Everything but the registered blink starts hidden.

Set `INSPECT_PAIR_IDX` to look at a different pair.

In [ ]:
INSPECT_PAIR_IDX  = 5
if not INSPECT_IN_NAPARI:
    print("napari inspection off (INSPECT_IN_NAPARI = False).")
else:
    try:
        import napari
    except ImportError as exc:
        raise ImportError(
            "napari is not installed in this kernel. "
        ) from exc

    pair = PAIRS[INSPECT_PAIR_IDX]
    a_raw = centre_crop(load_frame(pair["path1"]))
    b_raw = centre_crop(load_frame(pair["path2"]))
    a_reg, b_reg = get_registered(pair)
    shift = SHIFTS[pair["label"]]

    img_lims = tuple(np.percentile(a_reg, [1, 99]))
    diff_lims = tuple(np.percentile(a_raw - b_raw, [1, 99]))

    viewer = napari.Viewer(title=f"{pair['label']}  ({pair['px_nm']:.1f} nm/px)")
    viewer.add_image(np.stack([a_reg, b_reg]), name="registered (blink)",
                     contrast_limits=img_lims)
    viewer.add_image(np.stack([a_raw, b_raw]), name="unregistered (blink)",
                     contrast_limits=img_lims, visible=False)
    viewer.add_image(a_reg - b_reg, name="difference registered",
                     contrast_limits=diff_lims, visible=False)
    viewer.add_image(a_raw - b_raw, name="difference unregistered",
                     contrast_limits=diff_lims, visible=False)

    print(f"{pair['label']}   shift applied: dy={shift[0]:+.2f}, dx={shift[1]:+.2f} px")
    print("Drag the top slider on 'registered (blink)' - nothing should move.")
    print("Toggle 'unregistered (blink)' to see the drift that was corrected.")
    print("If no window appears, run  %gui qt  in a cell and re-run this one.")

### Optional - quantitative check

In [ ]:
if not SHOW_QUANT_CHECKS:
    print("Quantitative check off (SHOW_QUANT_CHECKS = False).")
else:
    diff_rows = []
    for p in PAIRS:
        s0 = (centre_crop(load_frame(p["path1"])) - centre_crop(load_frame(p["path2"]))).std()
        a_r, b_r = get_registered(p)
        s1 = (a_r - b_r).std()
        diff_rows.append({
            "label": p["label"], "px_nm": p["px_nm"],
            "drift_px": float(np.hypot(*SHIFTS[p["label"]])),
            "diff_std_before": s0, "diff_std_after": s1,
            "reduction_pct": 100.0 * (1.0 - s1 / s0),
        })
        print(f"  {p['label']:28s} {s0:8.0f} -> {s1:8.0f}   ({100.0 * (1.0 - s1 / s0):4.1f}% lower)")

    diff_df = pd.DataFrame(diff_rows)
    after = diff_df["diff_std_after"]
    print(f"\nPairs improved      : {int((diff_df['reduction_pct'] > 0).sum())}/{len(diff_df)}")
    print(f"Noise level after   : {after.min():.0f} - {after.max():.0f} "
          f"(spread {100.0 * (after.max() - after.min()) / after.mean():.0f}% of the mean)")
    print("A tight spread here means every pair collapsed onto the same detector noise floor.")
    display(diff_df)

## Step 3 - FRC machinery

In [ ]:
# The innermost rings hold almost no FFT samples - ring 0 holds the DC term alone, which mean
# subtraction has just set to zero, so its "correlation" is 0/0. Ring k holds roughly 2*pi*k
# samples, so this floor discards the first two or three rings only. Not a tuning knob.
MIN_RING_SAMPLES = 16


def apply_window(image, window=None):
    """Edge taper applied before the FFT. Default (`FRC_WINDOW`) is none - see the notes above.

    Kept as a switch so the sensitivity to that choice can be measured rather than assumed.
    """
    window = FRC_WINDOW if window is None else window
    if window in (None, "none"):
        return image
    ny, nx = image.shape
    if window == "hann":
        return image * np.outer(np.hanning(ny), np.hanning(nx))
    if window == "hamming":
        return image * np.outer(np.hamming(ny), np.hamming(nx))
    raise ValueError(f"Unknown window: {window}")


def radial_frequency_map(shape):
    """Radial spatial frequency (cycles / pixel) of every FFT sample, fftfreq convention."""
    fy = np.fft.fftfreq(shape[0])[:, None]
    fx = np.fft.fftfreq(shape[1])[None, :]
    return np.sqrt(fy ** 2 + fx ** 2)


# Ring maps are expensive to build and identical for every image of a given shape, so they are
# cached: this notebook computes thousands of FRC curves on a handful of distinct shapes.
_RING_CACHE = {}


def ring_index(shape, n_rings):
    """Ring binning for one FFT shape -> (keep mask, ring index of kept samples, centres, counts).

    Bin edges run 0 -> 0.5 cyc/px and every sample outside that radius is dropped. Those samples
    are the corners of the frequency square - 21% of the FFT - and they exist along the diagonals
    only, so a "ring" out there is four arcs whose correlation reports on the diagonal direction
    rather than on the image. Binning into them also yields 1/f < 2 px, a period finer than the
    sampling can represent, which is where spuriously good single-image numbers come from.
    """
    key = (tuple(shape), int(n_rings))
    if key not in _RING_CACHE:
        fmap = radial_frequency_map(shape).ravel()
        edges = np.linspace(0.0, 0.5, n_rings + 1)
        keep = fmap <= 0.5
        idx = np.minimum(np.searchsorted(edges, fmap[keep], side="right") - 1, n_rings - 1)
        centres = 0.5 * (edges[:-1] + edges[1:])
        n_pix = np.bincount(idx, minlength=n_rings).astype(np.float64)
        _RING_CACHE[key] = (keep, idx, centres, n_pix)
    return _RING_CACHE[key]


def frc_curve(image1, image2, n_rings=None, window=None):
    """Raw FRC between two equally-shaped 2-D images.

    Returns (frequencies, frc, n_pixels_per_ring). The frequencies are cycles per pixel *of the
    arrays passed in*; for checkerboard halves that pixel is two original pixels wide, which is
    what `px_scale` in `_package` converts back.
    """
    if image1.shape != image2.shape:
        raise ValueError(f"Images must match; got {image1.shape} vs {image2.shape}")

    # The mean is removed so that ring 0 measures structure. In an SEM image the DC level is
    # orders of magnitude above everything else and is trivially correlated between any two
    # frames, which would drag the innermost rings towards 1 for no physical reason.
    img1 = apply_window(np.asarray(image1, dtype=np.float64) - np.mean(image1), window)
    img2 = apply_window(np.asarray(image2, dtype=np.float64) - np.mean(image2), window)

    F1 = np.fft.fft2(img1)
    F2 = np.fft.fft2(img2)

    n_rings = (min(image1.shape) // 2) if n_rings is None else int(n_rings)
    keep, idx, centres, n_pix = ring_index(image1.shape, n_rings)

    num = np.bincount(idx, weights=np.real(F1 * np.conj(F2)).ravel()[keep], minlength=n_rings)
    p1 = np.bincount(idx, weights=(np.abs(F1) ** 2).ravel()[keep], minlength=n_rings)
    p2 = np.bincount(idx, weights=(np.abs(F2) ** 2).ravel()[keep], minlength=n_rings)

    den = np.sqrt(p1 * p2)
    frc_vals = np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)
    return centres, fill_undersampled(frc_vals, n_pix), n_pix


def fill_undersampled(curve, n_pix):
    """Replace the under-populated innermost rings with the first trustworthy value.

    Left alone they are 0/0 noise of order +/-1, and the smoothing below would spread that over
    the first few percent of the frequency axis.
    """
    curve = np.asarray(curve, dtype=np.float64).copy()
    valid = np.asarray(n_pix) >= MIN_RING_SAMPLES
    if valid.any() and not valid.all():
        curve[:int(np.argmax(valid))] = curve[int(np.argmax(valid))]
    return curve


def threshold_curve(n_pixels_per_ring, criterion=None):
    """Fixed thresholds ("0.143", "0.5") or the van Heel & Schatz (2005) information curves."""
    criterion = THRESHOLD if criterion is None else criterion
    n = np.maximum(np.asarray(n_pixels_per_ring, dtype=np.float64), 1.0)
    if criterion == "0.143":
        return np.full_like(n, 0.143)
    if criterion == "0.5":
        return np.full_like(n, 0.5)
    if criterion in ("half_bit", "halfbit"):
        snr = 0.2071
    elif criterion in ("one_bit", "onebit"):
        snr = 0.5
    else:
        raise ValueError(f"Unknown threshold criterion: {criterion}")
    sq = np.sqrt(n)
    return (snr + (2.0 * np.sqrt(snr) + 1.0) / sq) / (snr + 1.0 + 2.0 * np.sqrt(snr) / sq)


def smooth_window(n_rings, frac=None):
    """Odd moving-average width used for a curve of `n_rings` points.

    A fixed *fraction* rather than a fixed number of rings keeps the amount of smoothing
    comparable between a 2048 px frame (1024 rings) and a 256 px patch (64 rings), which matters
    because Step 6 fits a calibration at both scales.
    """
    frac = SMOOTH_FRAC if frac is None else frac
    return max(3, int(round(frac * n_rings)) | 1)


def smooth_curve(y, frac=None):
    """Moving average over `smooth_window` rings (reflected edges).

    Root-finding on the raw curve is unreliable: one noisy ring dipping under the threshold would
    be read as the resolution.
    """
    y = np.asarray(y, dtype=np.float64)
    w = smooth_window(len(y), frac)
    if w >= len(y):
        return y.copy()
    padded = np.pad(y, w // 2, mode="reflect")
    return np.convolve(padded, np.ones(w) / w, mode="valid")


def find_crossing(freqs, curve, thresh, n_pix, persist=None):
    """First frequency where *curve* falls below *thresh* and stays below -> (frequency, status).

    `persist` consecutive rings must also be below, so a single dip is not read as the
    resolution, and the numerically degenerate innermost rings (fewer than MIN_RING_SAMPLES
    samples) cannot host a crossing. Nothing beyond that is filtered: van Heel & Schatz (2005)
    would argue that a fixed 0.143 is not significant in a sparse ring either, but applying that
    as a cut would discard the coarse end of the axis and would no longer be the method Koho and
    Dumoux describe, which is the method this notebook exists to recalibrate.

    Two outcomes return NaN rather than a number, because in both there is no crossing to report
    and returning the end of the axis would invent one:

    - "beyond_floor" - the curve never crosses, so the resolution is finer than the finest period
      these samples can carry. The measurement is censored from below, not equal to the floor.
    - "no_signal"    - already below the threshold at the first eligible ring, i.e. no correlated
      structure anywhere on the measurable part of the axis.
    """
    freqs = np.asarray(freqs, dtype=np.float64)
    thresh = np.asarray(thresh, dtype=np.float64)
    n = freqs.size
    persist = max(3, n // 50) if persist is None else int(persist)

    usable = np.asarray(n_pix) >= MIN_RING_SAMPLES
    if not usable.any():
        return np.nan, "no_signal"
    i_min = int(np.argmax(usable))

    below = np.asarray(curve, dtype=np.float64) < thresh
    # Rolling "all below" over the next `persist` rings, via a cumulative count of the not-belows.
    csum = np.concatenate(([0], np.cumsum(~below)))
    stays = (csum[np.minimum(np.arange(n) + persist, n)] - csum[:n]) == 0
    stays[:i_min] = False

    if not stays.any():
        return np.nan, "beyond_floor"
    i = int(np.argmax(stays))
    if i == i_min:
        return np.nan, "no_signal"

    d_lo = curve[i - 1] - thresh[i - 1]
    d_hi = curve[i] - thresh[i]
    denom = d_lo - d_hi
    if abs(denom) < 1e-12:
        return float(freqs[i]), "ok"
    return float(freqs[i - 1] + (freqs[i] - freqs[i - 1]) * d_lo / denom), "ok"


print("FRC core ready: rings binned to Nyquist only, window = %r, root-finding on a curve "
      "smoothed over %.1f%% of the rings." % (FRC_WINDOW, 100 * SMOOTH_FRAC))

In [ ]:
def split_checkerboard(image, diagonal=0):
    """One of the two checkerboard sub-lattices of *image* (Koho et al. 2019).

    `diagonal=0` pairs the (even, even) pixels with the (odd, odd) ones; `diagonal=1` pairs
    (even, odd) with (odd, even). Either way each half is a square lattice of twice the original
    spacing, so its Nyquist period is 4 original pixels - the floor this whole method sits on.

    The two halves of a diagonal are offset from each other by one original pixel, i.e. half a
    pixel on the half-image grid. That offset alone multiplies the correlation by cos(pi*(uy+ux))
    and so drives it to zero towards the half-image Nyquist, whatever the image contains; it is
    the larger half of the checkerboard bias. Averaging the two diagonals makes what is left
    symmetric in x and y.
    """
    img = np.asarray(image, dtype=np.float64)
    if diagonal == 0:
        a, b = img[0::2, 0::2], img[1::2, 1::2]
    elif diagonal == 1:
        a, b = img[0::2, 1::2], img[1::2, 0::2]
    else:
        raise ValueError("diagonal must be 0 or 1")
    h = min(a.shape[0], b.shape[0])
    w = min(a.shape[1], b.shape[1])
    return a[:h, :w], b[:h, :w]


def _package(freqs, raw, n_pix, threshold, pixel_size_nm, split, px_scale, floor_px):
    """Turn a curve into a result dict: smoothing, threshold, crossing, resolution, flagging."""
    smoothed = smooth_curve(raw)
    thr = threshold_curve(n_pix, threshold)
    f_cross, status = find_crossing(freqs, smoothed, thr, n_pix)

    # `freqs` counts cycles per sample of whatever was correlated; px_scale (2 for the
    # checkerboard halves) puts it back on the original pixel grid.
    res_px = px_scale / f_cross if np.isfinite(f_cross) and f_cross > 0 else np.nan
    res_nm = res_px * pixel_size_nm if pixel_size_nm else np.nan

    # A crossing this close to the limit was found in the last few rings, where the smoothing
    # window reflects and the persistence rule runs out of axis. Flagged, not withheld: Step 8
    # refits without these points and reports whether excluding them changes the calibration.
    near_limit = bool(np.isfinite(res_px) and res_px < NYQUIST_FLAG_MULTIPLE * floor_px)

    unit = "nm" if pixel_size_nm else "px"
    scale = pixel_size_nm if pixel_size_nm else 1.0
    if status == "beyond_floor":
        res_str = f"<= {floor_px * scale:.2f} {unit} (never crosses; at the sampling limit)"
    elif status == "no_signal":
        res_str = "no correlated signal"
    elif near_limit:
        res_str = f"{res_px * scale:.2f} {unit} (flagged: within {NYQUIST_FLAG_MULTIPLE:g}x the limit)"
    else:
        res_str = f"{res_px * scale:.2f} {unit}"

    return {
        "split": split,
        "frequencies": freqs,                    # cycles per sample of the correlated arrays
        "freq_cyc_per_px": freqs / px_scale,     # cycles per pixel of the original image
        "frc_raw": raw,
        "frc_smooth": smoothed,
        "threshold_curve": thr,
        "threshold_name": threshold,
        "n_pixels_per_ring": n_pix,
        "crossing_freq": f_cross,
        "crossing_freq_orig": f_cross / px_scale if np.isfinite(f_cross) else np.nan,
        "status": status,
        "near_limit": near_limit,
        "resolution_px": res_px,
        "resolution_nm": res_nm,
        "resolution_str": res_str,
        "pixel_size_nm": pixel_size_nm,
        "px_scale": px_scale,
        "floor_px": floor_px,
    }


def frc_two_images(image1, image2, threshold=None, n_rings=None, pixel_size_nm=None, window=None):
    """Gold standard: FRC between two independent acquisitions of the same field of view.

    No split and no noise model - the two frames really are independent, so this is the reference
    everything else is calibrated against. Its floor is the ordinary Nyquist period, 2 px.
    """
    freqs, raw, n_pix = frc_curve(image1, image2, n_rings=n_rings, window=window)
    return _package(freqs, raw, n_pix, threshold or THRESHOLD, pixel_size_nm,
                    split="two_image", px_scale=1.0, floor_px=2.0)


def frc_checkerboard(image, threshold=None, n_rings=None, pixel_size_nm=None, window=None,
                     average_diagonals=True):
    """One-image FRC by checkerboard split, both diagonals averaged (Koho 2019 / Dumoux 2023).

    The two diagonal splits are independent estimates of the same curve, so averaging them halves
    the ring-to-ring noise as well as symmetrising the half-pixel offset. Koho's implementation
    averages them and Quoll inherits that, so we do too - the point here is to recalibrate their
    method, not to replace it.
    """
    curves = []
    for diagonal in ((0, 1) if average_diagonals else (0,)):
        a, b = split_checkerboard(image, diagonal)
        freqs, raw, n_pix = frc_curve(a, b, n_rings=n_rings, window=window)
        curves.append(raw)
    return _package(freqs, np.mean(curves, axis=0), n_pix, threshold or THRESHOLD, pixel_size_nm,
                    split="checkerboard", px_scale=2.0, floor_px=CHECKERBOARD_FLOOR_PX)


def calibration_model(r, a, b, c, d):
    """Dumoux/Quoll calibration form: r_ref = a * exp(c * (r - b)) + d.

    Maps a one-image checkerboard resolution onto the gold-standard scale. An exponential is the
    right shape because the bias is not a constant factor: as the true resolution worsens the
    checkerboard estimate saturates against its own 4 px floor, so the correction has to grow
    steeply at the top end.
    """
    return a * np.exp(c * (np.asarray(r, dtype=np.float64) - b)) + d


def fit_calibration(r_one, r_ref, b=None, n_scan=400):
    """Fit `calibration_model` to paired (one-image, gold-standard) resolutions.

    Two things make this fit well behaved without an optimiser:

    - `b` is fixed rather than fitted. `a * exp(-c*b)` is a single number, so a and b are not
      separately identifiable and letting both float sends a general-purpose optimiser wandering
      along that valley. Fixing b at the mean of `r_one` centres the exponent. The returned
      (a, b, c, d) still describe exactly the published four-parameter form.
    - With b fixed the model is *linear* in a and d for any given c, so the fit is a scan over
      the single parameter c with an exact least-squares solve for (a, d) inside it. That finds
      the global optimum instead of the nearest local one, and needs nothing but numpy.

    Returns (params, stats); stats holds n, rmse, r2 and the residual standard deviation.
    """
    r_one = np.asarray(r_one, dtype=np.float64)
    r_ref = np.asarray(r_ref, dtype=np.float64)
    ok = np.isfinite(r_one) & np.isfinite(r_ref)
    r_one, r_ref = r_one[ok], r_ref[ok]
    if r_one.size < 4:
        raise ValueError(f"need at least 4 finite pairs; got {r_one.size}")

    b = float(np.mean(r_one)) if b is None else float(b)
    span = max(float(np.ptp(r_one)), 1e-9)

    def solve(c):
        """Best (a, d) and its residual sum of squares for one value of c."""
        X = np.column_stack([np.exp(c * (r_one - b)), np.ones_like(r_one)])
        coef = np.linalg.lstsq(X, r_ref, rcond=None)[0]
        return coef, float(np.sum((X @ coef - r_ref) ** 2))

    # c is scanned geometrically over four decades either side of zero: c*span sets how much
    # curvature the exponential has across the data, and |c*span| > ~12 just overflows.
    grid = np.geomspace(1e-4, 12.0, n_scan // 2) / span
    grid = np.concatenate([-grid[::-1], grid])
    sse = np.array([solve(c)[1] for c in grid])
    k = int(np.argmin(sse))

    # Golden-section refinement inside the bracketing grid interval.
    lo, hi = grid[max(k - 1, 0)], grid[min(k + 1, grid.size - 1)]
    phi = 0.5 * (np.sqrt(5.0) - 1.0)
    x1, x2 = hi - phi * (hi - lo), lo + phi * (hi - lo)
    f1, f2 = solve(x1)[1], solve(x2)[1]
    for _ in range(60):
        if f1 < f2:
            hi, x2, f2 = x2, x1, f1
            x1 = hi - phi * (hi - lo)
            f1 = solve(x1)[1]
        else:
            lo, x1, f1 = x1, x2, f2
            x2 = lo + phi * (hi - lo)
            f2 = solve(x2)[1]
    c = 0.5 * (lo + hi)

    (a, d), ss_res = solve(c)
    params = (float(a), b, float(c), float(d))
    resid = r_ref - calibration_model(r_one, *params)
    ss_tot = float(np.sum((r_ref - r_ref.mean()) ** 2))
    stats = {
        "n": int(r_one.size),
        "rmse": float(np.sqrt(ss_res / r_one.size)),
        "resid_std": float(np.std(resid, ddof=1)) if r_one.size > 1 else np.nan,
        "r2": float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
    }
    return params, stats


def apply_calibration(r_one, params):
    """Put a one-image checkerboard resolution onto the gold-standard scale."""
    return calibration_model(r_one, *params)


def plot_frc(result, title="", ax=None, show_raw=True, color="steelblue", label=None):
    """FRC against frequency in cycles per *original* pixel, so one-image and two-image curves
    can share an axis despite the checkerboard halves being subsampled."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    f = result["freq_cyc_per_px"]
    if show_raw:
        ax.plot(f, result["frc_raw"], lw=0.7, alpha=0.3, color=color)
    ax.plot(f, result["frc_smooth"], lw=1.7, color=color,
            label=label or f"FRC ({result['split']})")
    ax.plot(f, result["threshold_curve"], "--", color="crimson", lw=1.1,
            label=f"threshold ({result['threshold_name']})")
    ax.axhline(0.0, color="grey", lw=0.8)

    floor_f = 1.0 / result["floor_px"]
    ax.axvspan(floor_f, 0.5, color="grey", alpha=0.12)
    ax.axvline(floor_f, color="grey", lw=1.0, ls="-.",
               label=f"sampling floor ({result['floor_px']:.0f} px)")
    if np.isfinite(result["crossing_freq_orig"]):
        ax.axvline(result["crossing_freq_orig"], color="green", ls=":", lw=1.5,
                   label=f"resolution = {result['resolution_str']}")

    ax.set_xlabel("Spatial frequency (cycles / original pixel)")
    ax.set_ylabel("FRC")
    ax.set_xlim(0, 0.5)
    ax.set_ylim(-0.2, 1.05)
    ax.set_title(title, fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    return ax


print("FRC API ready: frc_two_images, frc_checkerboard, fit_calibration, apply_calibration, plot_frc")

### One real pair

The same two measurements on a registered pair from the dataset, so the machinery is seen working on
microscope data before Step 4 runs it over everything. `INSPECT_PAIR_IDX` selects the pair.

In [ ]:
# A first look at one real pair. This is not the Step 4 measurement - it is here so the machinery
# is seen working on microscope data before it is run over the whole dataset.
pair = PAIRS[INSPECT_PAIR_IDX]
a_reg, b_reg = get_registered(pair)

ref = frc_two_images(a_reg, b_reg, threshold=THRESHOLD, n_rings=N_RINGS, pixel_size_nm=pair["px_nm"])
one = frc_checkerboard(a_reg, threshold=THRESHOLD, n_rings=N_RINGS, pixel_size_nm=pair["px_nm"])

print(f"{pair['label']}   ({pair['px_nm']:.1f} nm/px, {CROP_SIZE} px centre crop)")
print(f"  two-image, gold standard : {ref['resolution_str']:<52s} [{ref['status']}]")
print(f"  checkerboard, frame 1    : {one['resolution_str']:<52s} [{one['status']}]")
print(f"  sampling limits          : {2.0 * pair['px_nm']:.1f} nm two-image, "
      f"{CHECKERBOARD_FLOOR_PX * pair['px_nm']:.1f} nm checkerboard")
if np.isfinite(ref["resolution_nm"]) and np.isfinite(one["resolution_nm"]):
    print(f"  raw one-image / gold     : x{one['resolution_nm'] / ref['resolution_nm']:.2f}  "
          f"(one number from one pair - Step 6 fits the trend, it does not average ratios)")
if ref["near_limit"] or one["near_limit"]:
    print("  NOTE: at least one crossing sits in the last few rings, so it carries the "
          "`near_limit` flag.\n        It is still fitted; Step 8 checks whether that matters.")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
plot_frc(ref, title=f"two-image FRC - {pair['label']}", ax=axes[0], color="black")
plot_frc(one, title=f"checkerboard one-image FRC - {pair['label']}", ax=axes[1], color="darkorange")
axes[1].plot(ref["freq_cyc_per_px"], ref["frc_smooth"], color="black", lw=1.0, alpha=0.5,
             label="two-image, same pair")
axes[1].legend(loc="upper right", fontsize=8)
fig.suptitle(f"{pair['group']} - {pair['px_nm']:.1f} nm/px", fontsize=11)
fig.tight_layout()
plt.show()

## Step 4 - gold standard: two-image FRC (`r_ref`)

For every registered pair, the FRC between the two independent acquisitions. No split, no noise
model, no assumption about the detector - the frames really are independent, so this is the
reference the checkerboard measurement is calibrated against.

Measured at both scales, because Step 6 needs both:

- **full frame** - one `r_ref` per pair, from the whole `CROP_SIZE` crop;
- **patches** - one `r_ref` per 256 px patch, on the 8 x 8 grid that tiles the crop exactly.

Two things to read carefully in the output. Where the pixel size is too coarse the measurement
is **sampling-limited**: nothing finer than a 2 px period exists to be measured, the curve never
crosses the threshold, and `find_crossing` returns NaN with status `beyond_floor`. Those points
are absent rather than excluded - there is no number to exclude. At 5 nm/px in particular the
gold standard is expected to hit that limit. Measurements that do cross, but only in the last few
rings, are marked `ref_near_limit`; they stay in the fit, and Step 8 refits without them to test
whether the flag made any difference.

Patch measurements are also intrinsically noisier than the full frame: a 256 px patch has a
sixty-fourth of the Fourier samples, so its rings are correspondingly sparse. That scatter is
not a defect to be smoothed away - it is the quantity Step 6 reports as the smallest resolution
difference this method can actually detect.

In [ ]:
def patch_grid(n=None, size=None):
    """Top-left corners of the non-overlapping patch grid over an n x n crop."""
    n = CROP_SIZE if n is None else n
    size = PATCH_SIZE if size is None else size
    k = n // size
    return [(i * size, j * size) for i in range(k) for j in range(k)]


def patch_of(img, origin, size=None):
    size = PATCH_SIZE if size is None else size
    y0, x0 = origin
    return img[y0:y0 + size, x0:x0 + size]


PATCHES = patch_grid()

GOLD = {}                      # full-frame result objects, for plotting curves later
gold_rows, gold_patch_rows = [], []

t0 = time.time()
for pair in PAIRS:
    a, b = get_registered(pair)
    meta = {"group": pair["group"], "label": pair["label"], "px_nm": pair["px_nm"]}

    r = frc_two_images(a, b, threshold=THRESHOLD, n_rings=N_RINGS, pixel_size_nm=pair["px_nm"])
    GOLD[pair["label"]] = r
    gold_rows.append({**meta,
                      "r_ref_px": r["resolution_px"], "r_ref_nm": r["resolution_nm"],
                      "ref_status": r["status"], "ref_near_limit": r["near_limit"]})

    for k, origin in enumerate(PATCHES):
        rp = frc_two_images(patch_of(a, origin), patch_of(b, origin), threshold=THRESHOLD,
                            n_rings=N_RINGS, pixel_size_nm=pair["px_nm"])
        gold_patch_rows.append({**meta, "patch": k, "y0": origin[0], "x0": origin[1],
                                "r_ref_px": rp["resolution_px"], "r_ref_nm": rp["resolution_nm"],
                                "ref_status": rp["status"], "ref_near_limit": rp["near_limit"]})

    print(f"{pair['label']:28s} ({pair['px_nm']:.1f} nm/px)  full frame -> {r['resolution_str']}")

gold_df = pd.DataFrame(gold_rows)
gold_patch_df = pd.DataFrame(gold_patch_rows)

print(f"\n{len(gold_df)} full-frame and {len(gold_patch_df)} patch measurements "
      f"in {time.time() - t0:.0f} s\n")

for name, df in (("full frame", gold_df), (f"{PATCH_SIZE} px patches", gold_patch_df)):
    print(f"{name}:")
    for px, sub in df.groupby("px_nm"):
        got = sub[sub["ref_status"] == "ok"]
        gone = int((sub["ref_status"] == "beyond_floor").sum())
        quiet = int((sub["ref_status"] == "no_signal").sum())
        flagged = int(sub["ref_near_limit"].sum())
        if len(got):
            body = (f"r_ref = {got['r_ref_nm'].mean():6.2f} +/- {got['r_ref_nm'].std():.2f} nm "
                    f"({got['r_ref_px'].mean():.2f} px)")
        else:
            body = "nothing crossed the threshold"
        print(f"  {px:4.1f} nm/px  {len(got):4d}/{len(sub):4d} measured   {body}"
              f"   [{gone} never cross, {quiet} no signal, {flagged} flagged near the limit]")
    print()

## Step 5 - one-image checkerboard FRC (`r_co1`)

The same two scales, but now on **each frame separately**, with the checkerboard split standing
in for a second acquisition. This is the measurement being recalibrated: in use there is only
ever one image, and no gold standard to compare it with.

Both frames of every pair are measured, and Step 6 averages them - Dumoux et al. define `r_co1`
as the mean over the pair, against that pair's single `r_ref`. The per-frame values are kept in
the CSV so the frame-to-frame spread stays visible; it is a useful check, since two frames of
the same field should give the same answer.

The frames come from `get_registered()`, the same arrays Step 4 used, so that patch (i, j) means
the same piece of specimen in both measurements. The one-image FRC does not need registration,
but without it frame 2's patch grid would be offset from frame 1's by the drift. Applying the
shift is safe here because a Fourier phase ramp is unitary - every frequency is multiplied by a
number of modulus one - so it cannot correlate neighbouring pixels and cannot inflate a
checkerboard correlation.

Expect these values to be **coarser than `r_ref`**, and increasingly so towards the fine end.
Two known effects push them that way, both quantified in Step 3: each half keeps a quarter of
the pixels, and the halves are offset by one pixel along a diagonal. The 4 px floor is also
twice as far away as the gold standard's, so a frame that is genuinely well resolved can be
pinned there while `r_ref` is still measuring.

In [ ]:
ONE = {}                       # full-frame result objects, keyed by (label, frame)
one_rows, one_patch_rows = [], []

t0 = time.time()
for pair in PAIRS:
    meta = {"group": pair["group"], "label": pair["label"], "px_nm": pair["px_nm"]}
    shown = []

    for frame_no, img in zip((1, 2), get_registered(pair)):
        r = frc_checkerboard(img, threshold=THRESHOLD, n_rings=N_RINGS,
                             pixel_size_nm=pair["px_nm"])
        ONE[(pair["label"], frame_no)] = r
        one_rows.append({**meta, "frame": frame_no,
                         "r_co1_px": r["resolution_px"], "r_co1_nm": r["resolution_nm"],
                         "one_status": r["status"], "one_near_limit": r["near_limit"]})
        shown.append(r["resolution_str"])

        for k, origin in enumerate(PATCHES):
            rp = frc_checkerboard(patch_of(img, origin), threshold=THRESHOLD, n_rings=N_RINGS,
                                  pixel_size_nm=pair["px_nm"])
            one_patch_rows.append({**meta, "frame": frame_no, "patch": k,
                                   "y0": origin[0], "x0": origin[1],
                                   "r_co1_px": rp["resolution_px"], "r_co1_nm": rp["resolution_nm"],
                                   "one_status": rp["status"], "one_near_limit": rp["near_limit"]})

    print(f"{pair['label']:28s} ({pair['px_nm']:.1f} nm/px)  frame 1 -> {shown[0]:<28s} "
          f"frame 2 -> {shown[1]}")

one_df = pd.DataFrame(one_rows)
one_patch_df = pd.DataFrame(one_patch_rows)

print(f"\n{len(one_df)} full-frame and {len(one_patch_df)} patch measurements "
      f"in {time.time() - t0:.0f} s\n")

for name, df in (("full frame", one_df), (f"{PATCH_SIZE} px patches", one_patch_df)):
    print(f"{name}:")
    for px, sub in df.groupby("px_nm"):
        got = sub[sub["one_status"] == "ok"]
        gone = int((sub["one_status"] == "beyond_floor").sum())
        quiet = int((sub["one_status"] == "no_signal").sum())
        flagged = int(sub["one_near_limit"].sum())
        body = (f"r_co1 = {got['r_co1_nm'].mean():6.2f} +/- {got['r_co1_nm'].std():.2f} nm "
                f"({got['r_co1_px'].mean():.2f} px)") if len(got) else "nothing crossed"
        print(f"  {px:4.1f} nm/px  {len(got):4d}/{len(sub):4d} measured   {body}"
              f"   [{gone} never cross, {quiet} no signal, {flagged} flagged near the limit]")
    print()

# The two frames of a pair are repeat acquisitions of one field, so their one-image values should
# agree. A large spread here would mean the measurement is dominated by noise rather than by the
# specimen, and would undermine averaging them in Step 6.
wide = one_df.pivot_table(index="label", columns="frame", values="r_co1_nm")
if {1, 2}.issubset(wide.columns):
    delta = (wide[1] - wide[2]).abs()
    rel = 100 * delta / wide.mean(axis=1)
    print(f"Frame-to-frame agreement (full frame): median |f1 - f2| = {delta.median():.2f} nm "
          f"({rel.median():.1f}% of the value), worst {delta.max():.2f} nm")

## Step 6 - the calibration curves

In [ ]:
FULL_FRAME = "full_frame"
PATCH_SCALE = f"patch_{PATCH_SIZE}"
SCALES = [(FULL_FRAME, "full frame"), (PATCH_SCALE, f"{PATCH_SIZE} px patches")]


def pair_average(df, keys):
    """Collapse the two frames of a pair into one `r_co1`, as Dumoux et al. define it.

    A pair is only marked `one_ok` if *both* frames produced a crossing; averaging a number with
    a NaN would otherwise pass a half-measurement off as a whole one.
    """
    out = df.groupby(keys, as_index=False).agg(
        r_co1_px=("r_co1_px", "mean"),
        r_co1_nm=("r_co1_nm", "mean"),
        one_near_limit=("one_near_limit", "any"),
        n_ok=("one_status", lambda s: int((s == "ok").sum())),
        n_frames=("frame", "count"),
    )
    out["one_ok"] = out["n_ok"] == out["n_frames"]
    return out.drop(columns=["n_ok"])


keys = ["group", "label", "px_nm"]
calib_full = gold_df.merge(pair_average(one_df, keys), on=keys, how="inner")
calib_full["scale"] = FULL_FRAME
calib_full["patch"] = -1
calib_full["y0"] = -1
calib_full["x0"] = -1

calib_patch = gold_patch_df.merge(pair_average(one_patch_df, keys + ["patch"]),
                                  on=keys + ["patch"], how="inner")
calib_patch["scale"] = PATCH_SCALE

calib_df = pd.concat([calib_full, calib_patch], ignore_index=True, sort=False)
calib_df["ratio"] = calib_df["r_co1_px"] / calib_df["r_ref_px"]

# `usable` means "both measurements produced a number", and nothing more. No point is withheld
# for sitting near the sampling limit: that is the regime the calibration exists to correct, and
# Dumoux et al. define no such exclusion. `flagged` records the ones whose crossing fell in the
# last few rings, so the fit below can be repeated without them as a sensitivity check.
calib_df["usable"] = ((calib_df["ref_status"] == "ok") & calib_df["one_ok"]
                      & np.isfinite(calib_df["ratio"]))
calib_df["flagged"] = calib_df["ref_near_limit"] | calib_df["one_near_limit"]

print("Calibration points, by scale - these are two separate calibrations, not one:")
for key, name in SCALES:
    sc = calib_df[calib_df["scale"] == key]
    u = sc[sc["usable"]]
    print(f"\n  {name}: {len(u)} of {len(sc)} have a number"
          f"   ({int(u['flagged'].sum())} flagged near the limit, still fitted)")
    if len(u):
        print(f"      median ratio {u['ratio'].median():.3f}   "
              f"IQR [{u['ratio'].quantile(0.25):.3f} - {u['ratio'].quantile(0.75):.3f}]   "
              f"r_co1 spans {u['r_co1_px'].min():.2f} - {u['r_co1_px'].max():.2f} px")
    for px, gp in sc.groupby("px_nm"):
        up = gp[gp["usable"]]
        body = f"ratio = {up['ratio'].median():.3f}" if len(up) else "-"
        print(f"      {px:4.1f} nm/px  {len(up):5d} usable   {body}"
              f"   (no number: {int((gp['ref_status'] != 'ok').sum())} r_ref, "
              f"{int((~gp['one_ok']).sum())} r_co1)")


def fit_scale(df, drop_flagged=False):
    """Fit `r_ref = f(r_co1)` in *pixel* units for one scale, or None if too few points.

    Pixel units because the bias mechanisms in Step 3 are geometric: they scale with the sampling
    grid, not with the magnification. A curve in pixels is one curve for every pixel size, and the
    per-group residuals printed below are the test of whether that actually holds.
    """
    d = df[df["usable"] & (~df["flagged"] if drop_flagged else True)]
    if len(d) < 4:
        return None
    params, stats = fit_calibration(d["r_co1_px"].to_numpy(), d["r_ref_px"].to_numpy())
    stats["x_min"], stats["x_max"] = float(d["r_co1_px"].min()), float(d["r_co1_px"].max())
    return {"params": params, "stats": stats, "data": d}


FITS = {key: {"all": fit_scale(calib_df[calib_df["scale"] == key]),
              "unflagged": fit_scale(calib_df[calib_df["scale"] == key], drop_flagged=True)}
        for key, _ in SCALES}

print("\n" + "=" * 78)
print("FITTED CALIBRATIONS   r_ref = a * exp(c * (r_co1 - b)) + d      (r_co1, r_ref in pixels)")
print("=" * 78)
for key, name in SCALES:
    fit = FITS[key]["all"]
    print(f"\n{name}:")
    if fit is None:
        print("  fewer than 4 usable points - not fitted.")
        continue
    a, b, c, d_ = fit["params"]
    s = fit["stats"]
    print(f"  a = {a:9.4f}   b = {b:7.4f}   c = {c:9.4f}   d = {d_:9.4f}")
    print(f"  n = {s['n']}   RMSE = {s['rmse']:.3f} px   residual std = {s['resid_std']:.3f} px"
          f"   R2 = {s['r2']:.3f}")
    print(f"  valid over r_co1 = {s['x_min']:.2f} - {s['x_max']:.2f} px; outside that it is "
          f"extrapolation and is not drawn.")

    # Does one curve in pixel units serve all three magnifications? If it does, each pixel-size
    # group scatters about zero. A group sitting consistently off zero means the bias is not
    # purely geometric and the calibration is tied to the imaging conditions.
    d = fit["data"].copy()
    d["resid_px"] = d["r_ref_px"] - apply_calibration(d["r_co1_px"], fit["params"])
    print("  residual by pixel size (should straddle zero if one curve serves all):")
    for px, g in d.groupby("px_nm"):
        print(f"      {px:4.1f} nm/px  n = {len(g):4d}   mean {g['resid_px'].mean():+.3f} px   "
              f"std {g['resid_px'].std():.3f} px")

    alt = FITS[key]["unflagged"]
    if alt is None:
        print("  sensitivity check: too few points once the flagged ones are removed.")
    elif alt["stats"]["n"] == s["n"]:
        print("  sensitivity check: nothing was flagged, so there is nothing to test.")
    else:
        xs = np.linspace(s["x_min"], s["x_max"], 100)
        shift = float(np.max(np.abs(apply_calibration(xs, fit["params"])
                                    - apply_calibration(xs, alt["params"]))))
        print(f"  sensitivity check: dropping the {s['n'] - alt['stats']['n']} flagged points "
              f"moves the curve by at most {shift:.3f} px")
        print(f"      -> {'material; report both' if shift > s['resid_std'] else 'smaller than the residual scatter, so it did not matter'}")

In [ ]:
colours = dict(zip(sorted(calib_df["px_nm"].unique()),
                   ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]))

panels = [("r_co1_px", "ratio", "$r_{co1}$ (px)", "$r_{co1}\\ /\\ r_{ref}$",
           "Calibration in pixel units"),
          ("r_co1_nm", "ratio", "$r_{co1}$ (nm)", "$r_{co1}\\ /\\ r_{ref}$",
           "The same curve seen in physical units"),
          ("r_co1_px", "r_ref_px", "$r_{co1}$ (px)", "$r_{ref}$ (px)",
           "What the fit maps: $r_{ref} = f(r_{co1})$")]

# One row per scale, axes shared down each column. Sharing is the point: the two calibrations are
# fitted separately, so the only honest way to see whether they differ is on identical axes.
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5), sharex="col", sharey="col")
usable_all = calib_df[calib_df["usable"]]

for row, (key, name) in enumerate(SCALES):
    pts = usable_all[usable_all["scale"] == key]
    fit = FITS[key]["all"]
    alt = FITS[key]["unflagged"]
    big = key == FULL_FRAME
    size = 85 if big else 14

    # The curve is only ever drawn across the range that was actually measured. An exponential
    # extrapolates violently, and a line drawn past the data would be an invention.
    if fit is not None:
        xs = np.linspace(fit["stats"]["x_min"], fit["stats"]["x_max"], 300)
        ys = apply_calibration(xs, fit["params"])
        ok = ys > 0
        xs, ys = xs[ok], ys[ok]

    for col, (xcol, ycol, xlab, ylab, ttl) in enumerate(panels):
        ax = axes[row, col]
        for px, sub in pts.groupby("px_nm"):
            ax.scatter(sub[xcol], sub[ycol], s=size, alpha=0.9 if big else 0.4,
                       color=colours[px], edgecolors="black" if big else "none",
                       linewidths=0.6, zorder=3, label=f"{px:.1f} nm/px")

        flag = pts[pts["flagged"]]
        if len(flag):
            ax.scatter(flag[xcol], flag[ycol], s=size + 30, facecolors="none",
                       edgecolors="crimson", linewidths=0.8, alpha=0.8, zorder=4,
                       label="flagged near the limit (still fitted)")

        if fit is not None and len(xs):
            if col == 0:
                ax.plot(xs, xs / ys, color="black", lw=2.2, zorder=5, label="fitted calibration")
            elif col == 1:
                # One curve in pixel units becomes a family of curves in nanometres, one per
                # magnification. That fanning-out is the argument for calibrating in pixels.
                # Each is clipped to the r_co1 range its own group covers, so the panel does not
                # imply coverage at a magnification where none was measured.
                for px, sub in pts.groupby("px_nm"):
                    keep = (xs >= sub["r_co1_px"].min()) & (xs <= sub["r_co1_px"].max())
                    if keep.any():
                        ax.plot(xs[keep] * px, (xs / ys)[keep], color=colours[px], lw=1.8,
                                zorder=5, label=f"fitted, {px:.1f} nm/px")
            else:
                ax.plot(xs, ys, color="black", lw=2.2, zorder=5, label="fitted calibration")

            if alt is not None and alt["stats"]["n"] != fit["stats"]["n"] and col != 1:
                ya = apply_calibration(xs, alt["params"])
                ax.plot(xs, xs / ya if col == 0 else ya, color="black", lw=1.4, ls=":",
                        zorder=5, label="refit without flagged points")

        if ycol == "ratio":
            ax.axhline(1.0, color="crimson", ls="--", lw=1.2, label="no correction needed")
        else:
            lim = float(np.nanpercentile(usable_all[[xcol, ycol]].values, 99.5)) if len(usable_all) else 1.0
            ax.plot([0, lim], [0, lim], "--", color="crimson", lw=1.2, label="identity")
            ax.axhline(2.0, color="grey", ls="-.", lw=1.0, label="$r_{ref}$ limit (2 px)")
        if xcol.endswith("_px"):
            ax.axvline(CHECKERBOARD_FLOOR_PX, color="grey", ls=":", lw=1.2,
                       label=f"$r_{{co1}}$ limit ({CHECKERBOARD_FLOOR_PX:.0f} px)")

        ax.set_ylabel(ylab)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=6.5, loc="best")
        if row == 0:
            ax.set_title(ttl, fontsize=10)
        else:
            ax.set_xlabel(xlab)

for row, (key, name) in enumerate(SCALES):
    n = int((usable_all["scale"] == key).sum())
    fig.text(0.006, 0.72 - 0.45 * row, f"{name.upper()}  (n = {n})", rotation=90,
             va="center", fontsize=12, weight="bold")

fig.suptitle("Two calibrations, not one: one-image checkerboard FRC recalibrated against the "
             "two-image gold standard", fontsize=13)
fig.tight_layout(rect=[0.022, 0, 1, 0.965])
plt.show()

# The two curves are fitted separately, so the question this plot has to answer is whether that
# separation is doing any work. If the clouds overlap and the curves lie on top of each other,
# one calibration would serve both and the split is needless complexity.
print("Does the scale matter?")
for key, name in SCALES:
    u = usable_all[usable_all["scale"] == key]
    if len(u):
        print(f"  {name:16s} n = {len(u):4d}   median ratio {u['ratio'].median():.3f}   "
              f"scatter (std) {u['ratio'].std():.3f}")

both = [FITS[k]["all"] for k, _ in SCALES]
if all(f is not None for f in both):
    lo = max(f["stats"]["x_min"] for f in both)
    hi = min(f["stats"]["x_max"] for f in both)
    if hi > lo:
        xs = np.linspace(lo, hi, 200)
        gap = float(np.max(np.abs(apply_calibration(xs, both[0]["params"])
                                  - apply_calibration(xs, both[1]["params"]))))
        noise = max(f["stats"]["resid_std"] for f in both)
        print(f"  over the r_co1 range they share ({lo:.2f} - {hi:.2f} px) the two fitted curves "
              f"differ by up to {gap:.3f} px,\n  against a residual scatter of {noise:.3f} px "
              f"-> {'separate calibrations are justified' if gap > noise else 'one curve may well serve both'}.")

pat = usable_all[usable_all["scale"] == PATCH_SCALE]
if len(pat):
    spread = pat.groupby("label")["ratio"].std().median()
    print(f"\nPatch-to-patch scatter of the ratio within a single field, median std: {spread:.3f}."
          f"\nThat is the limit on how small a real difference this method can detect.")

# If r_co1 had saturated it would stop varying with r_ref and the points would pile up
# vertically. That, not a fixed distance from the sampling limit, is what would disqualify a
# point - so it is measured rather than assumed.
near = usable_all[usable_all["r_co1_px"] < 1.5 * CHECKERBOARD_FLOOR_PX]
if len(near) > 3:
    c = float(np.corrcoef(near["r_co1_px"], near["r_ref_px"])[0, 1])
    print(f"\nWithin 1.5x the r_co1 limit ({len(near)} points): corr(r_co1, r_ref) = {c:.2f}. "
          f"Near zero\nwould mean r_co1 has saturated there and carries no information.")

## Step 7 - CSV output

Everything measured so far, written out. Seven files, and the distinction between them matters:
the first five are records of what was measured, `calibration_points.csv` is the only one the
fit will read, and `calibration_summary.csv` is the human-readable digest.

| file | one row per | holds |
|---|---|---|
| `registration.csv` | pair | measured drift and post-registration residual |
| `gold_standard_frc_full_frame.csv` | pair | `r_ref` over the whole crop |
| `gold_standard_frc_patches.csv` | pair x patch | `r_ref` per 256 px patch |
| `single_image_frc_full_frame.csv` | pair x frame | `r_co1` per frame |
| `single_image_frc_patches.csv` | pair x frame x patch | `r_co1` per frame per patch |
| `calibration_points.csv` | pair (or pair x patch) | paired `r_co1`/`r_ref`, ratio, `usable` |
| `calibration_summary.csv` | pixel size x scale | counts, medians, spread |

Every row carries its `status` and `near_limit` flags rather than being filtered on the way out.
A file that has already dropped rows cannot be audited afterwards, and the count of measurements
that never crossed the threshold is itself a result - it is the record of how much of this
dataset the method could not measure at all.

In [ ]:
summary_rows = []
for scale, sc in calib_df.groupby("scale"):
    for px, gp in sc.groupby("px_nm"):
        u = gp[gp["usable"]]
        summary_rows.append({
            "scale": scale,
            "px_nm": px,
            "n_total": len(gp),
            "n_usable": int(gp["usable"].sum()),
            "n_flagged": int((gp["usable"] & gp["flagged"]).sum()),
            "n_ref_no_crossing": int((gp["ref_status"] == "beyond_floor").sum()),
            "n_ref_no_signal": int((gp["ref_status"] == "no_signal").sum()),
            "n_one_no_crossing": int((~gp["one_ok"]).sum()),
            "r_ref_nm_median": u["r_ref_nm"].median(),
            "r_co1_nm_median": u["r_co1_nm"].median(),
            "r_ref_px_median": u["r_ref_px"].median(),
            "r_co1_px_median": u["r_co1_px"].median(),
            "ratio_median": u["ratio"].median(),
            "ratio_iqr": u["ratio"].quantile(0.75) - u["ratio"].quantile(0.25),
            "ratio_std": u["ratio"].std(),
        })

summary_df = pd.DataFrame(summary_rows)

fit_rows = []
for key, name in SCALES:
    for variant, fit in FITS[key].items():
        if fit is None:
            continue
        a, b, c, d_ = fit["params"]
        fit_rows.append({"scale": key, "variant": variant, "units": "pixels",
                         "a": a, "b": b, "c": c, "d": d_,
                         **{k: fit["stats"][k] for k in
                            ("n", "rmse", "resid_std", "r2", "x_min", "x_max")}})
fits_df = pd.DataFrame(fit_rows)

print("=== FITTED CALIBRATIONS (r_ref = a*exp(c*(r_co1 - b)) + d, pixels) ===")
display(fits_df)
print("=== CALIBRATION SUMMARY ===")
display(summary_df)

if OUTPUT_PATH:
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    written = [
        ("registration.csv", reg_df),
        ("gold_standard_frc_full_frame.csv", gold_df),
        ("gold_standard_frc_patches.csv", gold_patch_df),
        ("single_image_frc_full_frame.csv", one_df),
        ("single_image_frc_patches.csv", one_patch_df),
        ("calibration_points.csv", calib_df),
        ("calibration_fits.csv", fits_df),
        ("calibration_summary.csv", summary_df),
    ]
    print(f"\nWriting to {OUTPUT_PATH}")
    for name, df in written:
        df.to_csv(os.path.join(OUTPUT_PATH, name), index=False)
        print(f"  {name:38s} {len(df):6d} rows x {df.shape[1]:2d} columns")
    print("\ncalibration_points.csv holds the measurements: `usable` marks the rows that "
          "produced a\nnumber, `flagged` those measured in the last few rings. "
          "calibration_fits.csv holds the\nfitted parameters, one row per scale, plus the "
          "`unflagged` refit used as the sensitivity\ncheck. Both curves are in pixel units - "
          "multiply by the pixel size to read them in nm.")
else:
    print("\nOUTPUT_PATH is empty - nothing written.")